In [ ]:
# Initial document loading


# Run our new Docling structured sectional parser with local LM Studio VLM integration
from backend.utils.doc_processor import process_document

FILE_PATH = "/home/preritubuntu/Downloads/Attentio_is_all_you_need.pdf"
OUTPUT_JSON_PATH = "/home/preritubuntu/RAG projext/parsed_sections_docling.json"

# Set the API URL of your local LM Studio instance if it is running
# (e.g. "http://localhost:1234/v1")
LM_STUDIO_URL = "http://localhost:1234/v1"

process_document(
    file_path=FILE_PATH,
    output_json_path=OUTPUT_JSON_PATH,
    vlm_url=LM_STUDIO_URL,  # Set to None if your LM Studio server is not running
    run_vlm_on_tables=True  # Set to True to summarize table crops
)    

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Converting document: Attentio_is_all_you_need.pdf with Docling...


[INFO] 2026-07-17 01:44:12,168 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-17 01:44:12,180 [RapidOCR] download_file.py:60: File exists and is valid: /home/preritubuntu/miniconda3/envs/langchain_env/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-17 01:44:12,181 [RapidOCR] main.py:63: Using /home/preritubuntu/miniconda3/envs/langchain_env/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-17 01:44:12,248 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-17 01:44:12,250 [RapidOCR] download_file.py:60: File exists and is valid: /home/preritubuntu/miniconda3/envs/langchain_env/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-17 01:44:12,251 [RapidOCR] main.py:63: Using /home/preritubuntu/miniconda3/envs/langchain_env/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-17 01:44:12,288 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Grouping document items sequentially...
Generating local VLM description for Figure 1...
Generating local VLM description for Figure 2...
Generating local VLM description for Figure 3...
Generating local VLM description for Table 1...
Generating local VLM description for Table 2...
Generating local VLM description for Table 3...
Generating local VLM description for Table 4...
Generating local VLM description for Figure 4...
Generating local VLM description for Figure 5...
Generating local VLM description for Figure 6...
Saving structured JSON to: /home/preritubuntu/RAG projext/parsed_sections_docling.json
Successfully processed Attentio_is_all_you_need.pdf. Extracted:
 - 27 sections/headings
 - 4 tables
 - 6 figures/pictures


In [2]:
## Converted json into list of docs and chunking also done
from backend.loaders.data_loader import load_docling_json
# Load and partition the structured JSON
docs = load_docling_json("/home/preritubuntu/RAG projext/parsed_sections_docling.json")

Loaded and partitioned parsed_sections_docling.json into 61 chunks:
 - Text chunks: 51
 - Table chunks: 4
 - Figure chunks: 6


In [3]:
docs

[Document(metadata={'type': 'text', 'doc_id': 'Attentio_is_all_you_need', 'filename': 'Attentio_is_all_you_need.pdf', 'heading': 'Front Matter / Document Header', 'page_start': 1, 'page_end': 1, 'chunk_id': 'Attentio_is_all_you_need_sec_Front_Matter_/_Document_Header_txt_1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.'),
 Document(metadata={'type': 'text', 'doc_id': 'Attentio_is_all_you_need', 'filename': 'Attentio_is_all_you_need.pdf', 'heading': 'Attention Is All You Need', 'page_start': 1, 'page_end': 1, 'chunk_id': 'Attentio_is_all_you_need_sec_Attention_Is_All_You_Need_txt_1'}, page_content='Ashish Vaswani ∗ Google Brain avaswani@google.com Noam Shazeer ∗ Google Brain noam@google.com\nLlion Jones ∗ Google Research llion@google.com Niki Parmar ∗ Google Research nikip@google.com Aidan N. Gomez ∗ † University of Toronto aidan@cs.toronto.edu 

In [4]:
#vector store
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
# 1. Load the OpenRouter API Key
load_dotenv("/home/preritubuntu/LANGCHAIN/.env")
# 2. Initialize the OpenAI Embedding Model (via OpenRouter)
# Note: You can change the dimensions to 512, 1024, or the default 1536.
embedding = OpenAIEmbeddings(
    model='openai/text-embedding-3-small',
    dimensions=1536,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
# 3. Initialize Chroma
# IMPORTANT: If you get a dimension mismatch error, delete your old 'my_chroma_db' 
# directory first, or change the collection_name/persist_directory.
vector_store = Chroma(
    embedding_function=embedding,
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [ ]:
chunk_id_list = []
for doc in docs:
    # print(doc.metadata["chunk_id"])
    chunk_id_list.append(doc.metadata["chunk_id"])

vector_store.add_documents(docs,ids = chunk_id_list)

In [5]:
from re import search
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain.memory import ConversationBufferMemory
from langchain_core.output_parsers import StrOutputParser
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker

parser = StrOutputParser()

# 1. Initialize your individual retrievers
# Example: BM25 for keyword-based retrieval
bm25_retriever = BM25Retriever.from_documents(docs)   ## combined info of all docs must be passed
bm25_retriever.k = 5

# Example: Vector store for semantic retrieval
vectorstore_retriever = vector_store.as_retriever(
    search_type = "mmr",     # this enables the MMR search
    search_kwargs = {"k":5,"lambda_mult":0.7}
)

# 2. Combine them into an EnsembleRetriever
# weights define the importance of each retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vectorstore_retriever],
    weights=[0.3, 0.7]
)

# Reranker model
# 3. Use the reranker model
reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")


query = "What is attention mechanism?"

# Invoke the hybrid retriever
documents = ensemble_retriever.invoke(query)

# Reranked scores
pairs = [(query,doc.page_content) for doc in documents]
scores = reranker_model.score(pairs)

# Taking top n results
compressor = CrossEncoderReranker(model=reranker_model, top_n=3)

# Combine them into the ContextualCompressionRetriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever  # Your hybrid retriever from earlier
)
# 4. Invoke it
# This single call retrieves, reranks, and returns the top 3 best documents!
final_docs = compression_retriever.invoke("What is self attention mechanism?")
# Print out the filtered, highest-quality documents
for doc in final_docs:
    print(f"Page: {doc.metadata.get('page_start', doc.metadata.get('page'))}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Page: 2
Page: 3
Page: 3


In [7]:
print(final_docs)

[Document(id='Attentio_is_all_you_need_sec_2_Background_txt_2', metadata={'chunk_id': 'Attentio_is_all_you_need_sec_2_Background_txt_2', 'doc_id': 'Attentio_is_all_you_need', 'page_start': 2, 'filename': 'Attentio_is_all_you_need.pdf', 'page_end': 2, 'heading': '2 Background', 'type': 'text'}, page_content='Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].\nEnd-to-end memory networks are based on a recurrent attention mechanism instead of sequencealigned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [34].\nTo the best of our knowledge, however, the Transformer is the first transduct

In [5]:
full_text = ""
for i,doc in enumerate(final_docs):
    full_text += f"Information {i+1} : {doc.page_content} \n" 





prompt = f"Providing you the information about the {query} , and here is the Full information from which you have to answer\n {full_text} \n , do not add your own thinking in the answer , answer only on the basis of the provided info "


import os
import dotenv
from langchain_openai import ChatOpenAI

# load_dotenv(dotenv_path=Path(__file__).resolve().parent.parent.parent / ".env")
load_dotenv(dotenv_path = "/home/preritubuntu/RAG projext/.env")

# Initialize ChatOpenAI pointing to OpenRouter
model = ChatOpenAI(
    model="qwen/qwen3.7-plus",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    # temperature = 0,   # randomness of language models output
    max_completion_tokens = 1000
)

result = model.invoke(prompt)
print(result.content)

Providing you the information about the What is attention mechanism? , and here is the Full information from which you have to answer
 Information 1 : Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].
End-to-end memory networks are based on a recurrent attention mechanism instead of sequencealigned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [34].
To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequencealigned RNNs or convolution. In the follow

In [6]:
type(embedding)

langchain_openai.embeddings.base.OpenAIEmbeddings

In [6]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    openai_api_key="lm-studio",
    model="google/gemma-3-4b",
)

In [7]:
query = "hello how are you"
model.invoke(query)

AIMessage(content="I’m doing well, thank you for asking! As a large language model, I don't really *feel* in the same way humans do, but my systems are running smoothly and I’m ready to chat. 😊 \n\nHow about you? How is your day going so far?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 13, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-tp13mpmu2c9cmcl2zn79', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f6a71-b9cc-7962-9049-08f1e0521585-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 62, 'total_tokens': 75, 'input_token_details': {}, 'output_token_details': {'reasoning': 0}})

7.5
